# Lab 2 · Exercise B — The on/off problem
A counting experiment with an auxiliary measurement of the background:
$n_{\rm on}=17$ events in the signal region, $n_{\rm off}=40$ in the control region, $\tau=10.53$.

$$L(s,b)=\text{Pois}(n_{\rm on}\,|\,s+b)\;\text{Pois}(n_{\rm off}\,|\,\tau b)$$

In [1]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 1281521   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Build and fit
Build the model with two `RooPoisson` and a `RooProdPdf`, import the single observation, fit, and read $\hat s$ and $\hat b$.
Hint: the means are `RooFormulaVar("mu_on", "@0+@1", ...)` and `RooFormulaVar("mu_off", "@0*@1", ...)`.

In [2]:
non, noff, tau_val = 17, 40, 10.53

s   = ROOT.RooRealVar("s", "signal", 10, 0, 200)
b   = ROOT.RooRealVar("b", "background", 4, 0.01, 200)
tau = ROOT.RooRealVar("tau", "tau", tau_val); tau.setConstant(True)
n_on  = ROOT.RooRealVar("n_on", "n_on", 0, 300)
n_off = ROOT.RooRealVar("n_off", "n_off", 0, 600)

mu_on  = ROOT.RooFormulaVar("mu_on", "@0+@1", ROOT.RooArgList(s,b))   # TODO: s + b, use RooFormulaVar(..)  
mu_off = ROOT.RooFormulaVar("mu_off", "@0*@1", ROOT.RooArgList(tau,b))    # TODO: tau * b, use RooFormulaVar(..)
p_on   = ROOT.RooPoisson("p_on", "p_on", n_on, mu_on)  # TODO: RooPoisson(n_on, mu_on)
p_off  = ROOT.RooPoisson("p_off", "p_off", n_off, mu_off)   # TODO: RooPoisson(n_off, mu_off)
pdf    = ROOT.RooProdPdf("pdf", "pdf", ROOT.RooArgList(p_on, p_off))   # TODO: product of the two, use RooProdPdf(..)

d = ROOT.RooDataSet("d", "d", ROOT.RooArgSet(n_on, n_off))
n_on.setVal(non); n_off.setVal(noff)
d.add(ROOT.RooArgSet(n_on, n_off))

nll   = pdf.createNLL(d)    # TODO: pdf.createNLL(d)
minim = ROOT.RooMinimizer(nll)
minim.setPrintLevel(-1)
minim.minimize("Minuit2", "Migrad")
minim.hesse()   # TODO: RooMinimizer(nll); set print level -1; minimize("Minuit2", "Migrad"); hesse()
print("s_hat = %.2f +- %.2f   b_hat = %.2f +- %.2f" % (s.getVal(), s.getError(), b.getVal(), b.getError()))

s_hat = 13.21 +- 4.16   b_hat = 3.80 +- 0.60


In [3]:
check("model and NLL built", pdf is not None and nll is not None and minim is not None)
check("b_hat close to n_off/tau = 3.80", abs(b.getVal() - noff / tau_val) < 0.2)
check("s_hat close to n_on - b_hat = 13.2", abs(s.getVal() - (non - noff / tau_val)) < 0.3)
check("error on b close to sqrt(n_off)/tau = 0.60", abs(b.getError() - np.sqrt(noff) / tau_val) < 0.1)

PASS  model and NLL built
PASS  b_hat close to n_off/tau = 3.80
PASS  s_hat close to n_on - b_hat = 13.2
PASS  error on b close to sqrt(n_off)/tau = 0.60


## B2 · Profile interval vs MINOS

Compute the 68 % interval on $s$ in two ways: scan the **profile** NLL until it rises by 0.5, and run **MINOS** on `s`.

*Concept Check*: MINOS *is* the numerical implementation of the profile likelihood scan. The intervals you get from the scan and from MINOS should be identical. Compare them with the symmetric HESSE errors.

`scan_interval` is provided: it needs a function returning $-\ln L_{\rm profile}(s)-(-\ln L)_{\min}$.



In [6]:
nll_min = nll.getVal()
s_hat, b_hat = s.getVal(), b.getVal()
prof = nll.createProfile(ROOT.RooArgSet(s))

def scan_interval(fn, xmax=60, n=600):
    xs = np.linspace(0.0, xmax, n)
    ys = np.array([fn(x) for x in xs])
    ok = xs[ys <= 0.5]
    return ok.min(), ok.max()

def prof_val(x):
    s.setVal(x)
    return prof.getVal()   # TODO: return the profile NLL at s=x minus nll_min   (prof.getVal() - nll_min)

lo_p, hi_p = scan_interval(prof_val)
print("profile : s in [%.2f, %.2f]  ->  %+.2f / %+.2f" % (lo_p, hi_p, lo_p - s_hat, hi_p - s_hat))

s.setVal(s_hat); b.setVal(b_hat)
# TODO: run MINOS on s only, then print s.getErrorLo() and s.getErrorHi()
minim.minos(ROOT.RooArgSet(s))
print("MINOS: s = %.2f  %+.2f / %+.2f" % (s.getVal(), s.getErrorLo(), s.getErrorHi()))

profile : s in [9.42, 17.63]  ->  -3.80 / +4.41
MINOS: s = 13.21  -3.86 / +4.49


In [7]:
check("profile interval contains the best fit", lo_p < s_hat < hi_p)
check("MINOS agrees with the profile scan (lower)", abs(s.getErrorLo() - (lo_p - s_hat)) < 0.3)
check("MINOS agrees with the profile scan (upper)", abs(s.getErrorHi() - (hi_p - s_hat)) < 0.3)

PASS  profile interval contains the best fit
PASS  MINOS agrees with the profile scan (lower)
PASS  MINOS agrees with the profile scan (upper)


In [8]:
# Plot profile
frame_prof = s.frame(ROOT.RooFit.Title("Profile Likelihood for s"))
prof.plotOn(frame_prof, ROOT.RooFit.ShiftToZero())

# Add 0.5 line
line = ROOT.TLine(s.getMin(), 0.5, s.getMax(), 0.5)
line.SetLineColor(ROOT.kGreen)
line.SetLineStyle(ROOT.kDotted)

c = ROOT.TCanvas("c", "c", 800, 500)
frame_prof.Draw()
line.Draw("SAME")
c.Draw()


[#0] ERROR:InputArguments -- RooArgSet::checkForDup: ERROR argument with name s is already in this set


### B3: The Impact of the Scale Factor $\tau$ (OPTIONAL)
The scale factor $\tau$ represents the relative exposure of the control region to the signal region. A larger $\tau$ means a more precise background estimate.
Let's see what happens if we degrade our control region. Keep the expected background $\hat{b}$ roughly the same, but change the observation to $n_{\text{off}} = 4$ and $\tau = 1.053$.

*Task*: Re-run the fit and plot the new Profile Likelihood for $s$ alongside the previous one. Observe how the interval widens.

In [9]:
# Update the RooPoisson for the off region with new values
noff_new = 4
tau_new = 1.053

# TODO: Re-run the fit ...
tau.setVal(tau_new)
n_off.setVal(noff_new)
d_new = ROOT.RooDataSet("d_new", "d_new", ROOT.RooArgSet(n_on, n_off))
d_new.add(ROOT.RooArgSet(n_on, n_off))
nll_new = pdf.createNLL(d_new)
minim_new = ROOT.RooMinimizer(nll_new)
minim_new.setPrintLevel(-1)
minim_new.minimize("Minuit2", "Migrad")
minim_new.hesse()
minim_new.minos(ROOT.RooArgSet(s))
print("New fit: s_hat = %.2f +- %.2f (MINOS: %+.2f / %+.2f), b_hat = %.2f +- %.2f" % 
      (s.getVal(), s.getError(), s.getErrorLo(), s.getErrorHi(), b.getVal(), b.getError()))
prof_new = nll_new.createProfile(ROOT.RooArgSet(s))
# Overlay comparison of profiles
frame_comp = s.frame(ROOT.RooFit.Range(0, 35), ROOT.RooFit.Title("Profile Likelihood: tau=10.53 (blue) vs tau=1.053 (red)"))
prof.plotOn(frame_comp, ROOT.RooFit.ShiftToZero(), ROOT.RooFit.LineColor(ROOT.kBlue))
prof_new.plotOn(frame_comp, ROOT.RooFit.ShiftToZero(), ROOT.RooFit.LineColor(ROOT.kRed))
line_comp = ROOT.TLine(0, 0.5, 35, 0.5)
line_comp.SetLineColor(ROOT.kGreen)
line_comp.SetLineStyle(ROOT.kDotted)
c_comp = ROOT.TCanvas("c_comp", "Comparison", 800, 500)
frame_comp.Draw()
line_comp.Draw("SAME")
c_comp.Draw()

New fit: s_hat = 13.20 +- 4.53 (MINOS: -4.36 / +4.79), b_hat = 3.80 +- 1.90
[#0] ERROR:InputArguments -- RooArgSet::checkForDup: ERROR argument with name s is already in this set


## Questions
Write your answer under each question, in 2–3 lines, and quote the numbers or plots of the steps indicated.
1. **(B3)** Compare the slice and the profile interval for $\tau=10.53$ and for $\tau=1.053$. Why are they almost equal in the first case and different in the second? Think of how well the control region measures `b`.
- For $\tau=10.53$, the control region measures $b$ with very high precision $\sigma_b \approx \sqrt{40}/10.53 \approx 0.60$, so profiling over $b$ barely shifts it, making slice and profile almost identical. For $\tau=1.053$, profiling lets $b$ adjust freely, widening the profile likelihood significantly relative to the slice.
2. **(B1)** Which parameter or number of the model carries the uncertainty on the background? Which measurement, and which Poisson count, does it come from?
- $\sigma_{b} \approx \sqrt{n_{off}}/\tau$. It comes directly from the auxiliary control region counting measurement $n_{off}$ modeled by Possion distribution
3. In a real analysis the control region is often a simulation or a sideband fit, not a counting measurement. Which part of the model of B1 would you change, and how?
- Replace the model from Poisson with a Gaussian constraint.